## Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 60)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline   

RANDOM_STATE   = 42
PARQUET_RAW    = "data/paysim_raw.parquet"
FRAUD_PALETTE  = {0: "#2196F3", 1: "#F44336"}

np.random.seed(RANDOM_STATE)
print("Imports and configuration ready.")

In [ ]:
assert os.path.exists(PARQUET_RAW), (
    f"❌ {PARQUET_RAW} not found. "
    f"Run Cell 21 in fraud_detection_stage1_2.ipynb first."
)

df = pd.read_parquet(PARQUET_RAW)

print(f" Loaded: {len(df):,} rows × {df.shape[1]} columns "
      f"({df.memory_usage(deep=True).sum() / 1024**2:.1f} MB)")

In [ ]:
df.shape

## Data Cleaning

In [ ]:
FRAUD_TYPES = ["TRANSFER", "CASH_OUT"]

before_rows  = len(df)
before_fraud = int(df["isFraud"].sum())

df_clean = df[df["type"].isin(FRAUD_TYPES)].copy()

after_rows  = len(df_clean)
after_fraud = int(df_clean["isFraud"].sum())

print("Filter: TRANSFER + CASH_OUT only")
print("─" * 48)
print(f"  Rows before  : {before_rows:>10,}")
print(f"  Rows removed : {before_rows - after_rows:>10,}  ({(before_rows - after_rows)/before_rows*100:.1f}% of dataset)")
print(f"  Rows kept    : {after_rows:>10,}")
print(f"  Fraud before : {before_fraud:>10,}")
print(f"  Fraud kept   : {after_fraud:>10,}")
print(f"  Fraud lost   : {before_fraud - after_fraud:>10,}  ← must be 0")
print("─" * 48)

# Hard assertion — if this fails something is wrong with the data or filter
assert after_fraud == before_fraud, "🚨 Fraud rows were lost — investigate the filter!"
print(" Zero fraud rows lost. Filter is safe to proceed with.")

In [ ]:
n_legit = int((df_clean["isFraud"] == 0).sum())
n_fraud = int((df_clean["isFraud"] == 1).sum())
fraud_rate_pct   = n_fraud / len(df_clean) * 100
imbalance_ratio  = n_legit / n_fraud

type_breakdown = (
    df_clean.groupby("type", observed=True)["isFraud"]
    .agg(fraud_count="sum", total_count="count")
    .assign(fraud_rate_pct=lambda x: (x["fraud_count"] / x["total_count"] * 100).round(4))
)

print("Class Distribution — df_clean (TRANSFER + CASH_OUT only)")
print("─" * 55)
print(f"  Legitimate (0) : {n_legit:>8,}  ({100 - fraud_rate_pct:.4f}%)")
print(f"  Fraud (1)      : {n_fraud:>8,}  ({fraud_rate_pct:.4f}%)")
print(f"  Imbalance ratio: {imbalance_ratio:.1f} : 1  (legit to fraud)")
print()
print("  Breakdown by transaction type:\n")
print(type_breakdown.to_string())
print()
print(f"   Note for Stage 7 → scale_pos_weight ≈ {round(imbalance_ratio)}")
print(f"     (tells XGBoost each fraud row is worth ~{round(imbalance_ratio)} legitimate rows)")

In [ ]:
n = len(df_clean)

print("Zero-Balance Counts in df_clean\n")
zero_inventory = {
    "newbalanceOrig == 0  (origin drained after tx)":  (df_clean["newbalanceOrig"] == 0).sum(),
    "oldbalanceOrg == 0   (origin was empty before)":  (df_clean["oldbalanceOrg"] == 0).sum(),
    "oldbalanceDest == 0  (dest empty before tx)":     (df_clean["oldbalanceDest"] == 0).sum(),
    "newbalanceDest == 0  (dest empty after tx)":      (df_clean["newbalanceDest"] == 0).sum(),
}
for desc, count in zero_inventory.items():
    print(f"  {desc}: {count:>8,}  ({count/n*100:>5.2f}%)")

# Compare rates by fraud label — the key diagnostic
print("\n  Zero-balance rates compared by class:")
print("  " + "─" * 65)
print(f"  {'Pattern':<38} {'Legitimate (%)':>14} {'Fraud (%)':>10}")
print("  " + "─" * 65)

patterns = [
    ("newbalanceOrig == 0  (origin drained)",   "newbalanceOrig"),
    ("oldbalanceDest == 0  (dest pre-empty)",   "oldbalanceDest"),
    ("newbalanceDest == 0  (dest post-empty)",  "newbalanceDest"),
]
for desc, colname in patterns:
    r_legit = (df_clean[df_clean["isFraud"] == 0][colname] == 0).mean() * 100
    r_fraud  = (df_clean[df_clean["isFraud"] == 1][colname] == 0).mean() * 100
    print(f"  {desc:<38} {r_legit:>14.2f} {r_fraud:>10.2f}")

print()
print("  → Large gaps between Legitimate and Fraud = discriminating signal.")
print("  → Stage 5 encodes each pattern as a binary flag feature.")
print("  → ZEROS ARE KEPT — they are features, not missing data.")

In [ ]:
df_clean = df_clean.drop(columns=["nameOrig", "nameDest"])

print("Dropped: nameOrig, nameDest")
print(f"Remaining columns ({df_clean.shape[1]}): {list(df_clean.columns)}")

In [ ]:
df_clean = df_clean.drop(columns=["isFlaggedFraud"])

df_clean["type"] = df_clean["type"].cat.remove_unused_categories()

print("Dropped: isFlaggedFraud")
print(f"Cleaned: 'type' category index now contains only: {df_clean['type'].unique().tolist()}")
print(f"\nRemaining columns ({df_clean.shape[1]}): {list(df_clean.columns)}")
print(f"\n  Target    : isFraud  (0 = legitimate, 1 = fraud)")
print(f"  Features  : all other columns (to be engineered in Stage 5)")
print(f"\n  Leakage risk:  eliminated")

In [ ]:
df_clean = df_clean.reset_index(drop=True)

mem_clean = df_clean.memory_usage(deep=True).sum() / 1024**2

print("=" * 62)
print("  STAGE 3 COMPLETE — df_clean SUMMARY")
print("=" * 62)
print(f"  Shape             : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"  Memory footprint  : {mem_clean:.1f} MB")
print(f"  Transaction types : {df_clean['type'].unique().tolist()}")
print(f"  Fraud count       : {df_clean['isFraud'].sum():,}")
print(f"  Fraud rate        : {df_clean['isFraud'].mean() * 100:.4f}%")
print(f"  Columns retained  : {list(df_clean.columns)}")
print()
print("  Cleaning decisions made:")
print("   Filtered to TRANSFER + CASH_OUT  (zero fraud rows lost)")
print("   Dropped nameOrig, nameDest        (high cardinality — no generalizable signal)")
print("   Dropped isFlaggedFraud            (target leakage)")
print("   Zero-balance rows KEPT            (they are informative signal for Stage 5)")
print("=" * 62)
print("\n  NEXT: Stage 4 — EDA on df_clean")

df_clean.head()

## Exploratory Data Analysis

In [ ]:
n_legit_c = int((df_clean["isFraud"] == 0).sum())
n_fraud_c = int((df_clean["isFraud"] == 1).sum())
fraud_rate_c = n_fraud_c / len(df_clean) * 100

print("Class balance in df_clean (filtered dataset):")
print(f"  Legitimate : {n_legit_c:>8,}  ({100 - fraud_rate_c:.4f}%)")
print(f"  Fraud      : {n_fraud_c:>8,}  ({fraud_rate_c:.4f}%)")
print()
print(f"  'Never-fraud' accuracy : {100 - fraud_rate_c:.3f}%  ← useless floor, not a goal")
print(f"  Primary metric         : PR-AUC  (Average Precision)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Left: bar chart with percentage annotations ───────────────────────────────
counts = [n_legit_c, n_fraud_c]
labels = ["Legitimate (0)", "Fraud (1)"]
bars   = axes[0].bar(labels, counts,
                     color=[FRAUD_PALETTE[0], FRAUD_PALETTE[1]], alpha=0.87)

for bar, pct in zip(bars, [100 - fraud_rate_c, fraud_rate_c]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 0.5,
        f"{pct:.3f}%", ha="center", va="center",
        color="white", fontweight="bold", fontsize=11,
    )

axes[0].set_title("Class Distribution\n(Filtered: TRANSFER + CASH_OUT)", fontweight="bold")
axes[0].set_ylabel("Transaction Count")
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)

# ── Right: exploded pie to emphasise rarity ───────────────────────────────────
axes[1].pie(
    counts,
    labels=labels,
    colors=[FRAUD_PALETTE[0], FRAUD_PALETTE[1]],
    autopct="%1.3f%%",
    startangle=90,
    explode=(0, 0.12),
    shadow=True,
    textprops={"fontsize": 10},
)
axes[1].set_title("Fraud Share — Exploded Slice", fontweight="bold")

plt.suptitle("Extreme Class Imbalance — The Core Modelling Challenge",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row_idx, tx_type in enumerate(["TRANSFER", "CASH_OUT"]):
    for col_idx, (fraud_label, fraud_name) in enumerate([(0, "Legitimate"), (1, "Fraud")]):
        subset = df_clean[
            (df_clean["type"] == tx_type) &
            (df_clean["isFraud"] == fraud_label)
        ]["amount"]

        ax = axes[row_idx, col_idx]
        ax.hist(subset, bins=60, color=FRAUD_PALETTE[fraud_label], alpha=0.78, density=True)
        ax.set_xscale("log")

        median_val = float(subset.median())
        ax.axvline(median_val, color="black", linestyle="--", linewidth=1.5,
                   label=f"Median: {median_val:,.0f}")

        ax.set_title(f"{tx_type}  —  {fraud_name}  (n={len(subset):,})", fontweight="bold")
        ax.set_xlabel("Amount (log scale)")
        ax.set_ylabel("Density")
        ax.legend(fontsize=9)

plt.suptitle(
    "Transaction Amount by Type and Fraud Label\n"
    "(Fraudulent amounts tend to cluster at higher values — one-shot account drains)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Tabular summary
print("Median and mean amount by type + fraud label:\n")
amount_stats = (
    df_clean.groupby(["type", "isFraud"], observed=True)["amount"]
    .agg(n="count", median="median", mean="mean")
    .round(2)
)
amount_stats.index = amount_stats.index.map(
    lambda x: f"{x[0]} — {'Fraud' if x[1] == 1 else 'Legit'}"
)
print(amount_stats.to_string())

In [ ]:
print("THE ACCOUNTING IDENTITY — Summary")
print("=" * 60)
print()
print("  ORIGIN (sender):")
print("    errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg")
print("    Clean transaction  →  ≈ 0  (books balance)")
print("    Fraudulent tx      →  often ≠ 0  (books tampered)")
print()
print("  DESTINATION (receiver):")
print("    errorBalanceDest = oldbalanceDest + amount - newbalanceDest")
print("    Clean transaction  →  ≈ 0  (books balance)")
print("    Fraudulent tx      →  often ≠ 0  (books tampered)")
print()
print("  These two features will be the strongest predictors in Stage 8.")
print("  Cells 12–15 show the empirical evidence.")

In [ ]:
_eda = df_clean.copy()

# Origin error: newbalanceOrig + amount − oldbalanceOrg should equal 0 if clean
_eda["errorBalanceOrig"] = (
    _eda["newbalanceOrig"] + _eda["amount"] - _eda["oldbalanceOrg"]
)

# Destination error: oldbalanceDest + amount − newbalanceDest should equal 0 if clean
_eda["errorBalanceDest"] = (
    _eda["oldbalanceDest"] + _eda["amount"] - _eda["newbalanceDest"]
)

print("_eda (temporary EDA copy) created — df_clean is unchanged.")
print()
print("Error feature formulas:")
print("  errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg")
print("  errorBalanceDest = oldbalanceDest + amount - newbalanceDest")
print()

# Quick discrimination preview: how different are fraud vs legit error magnitudes?
print("Mean absolute error by class (higher ratio = stronger discrimination):\n")
print(f"  {'Feature':<22} {'Legit MAE':>14} {'Fraud MAE':>14} {'Ratio (F/L)':>12}")
print("  " + "─" * 66)
for ecol in ["errorBalanceOrig", "errorBalanceDest"]:
    legit_mae = float(np.abs(_eda[_eda["isFraud"] == 0][ecol]).mean())
    fraud_mae = float(np.abs(_eda[_eda["isFraud"] == 1][ecol]).mean())
    ratio     = fraud_mae / legit_mae if legit_mae > 0 else float("inf")
    print(f"  {ecol:<22} {legit_mae:>14,.2f} {fraud_mae:>14,.2f} {ratio:>11.1f}×")

print()
print("  → A large ratio means fraud errors are much larger than legit errors.")
print("  → Even a ratio of 5× or 10× is a very strong feature signal.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, ecol in enumerate(["errorBalanceOrig", "errorBalanceDest"]):

    legit_vals = _eda.loc[_eda["isFraud"] == 0, ecol].values
    fraud_vals = _eda.loc[_eda["isFraud"] == 1, ecol].values

    # ── Left: coloured box plot (matplotlib — version-independent) ────────────
    ax_box = axes[row_idx, 0]
    bp = ax_box.boxplot(
        [legit_vals, fraud_vals],
        labels=["Legitimate", "Fraud"],
        patch_artist=True,
        showfliers=False,
        widths=0.5,
    )
    # Colour each box using FRAUD_PALETTE
    for patch, colour in zip(bp["boxes"], [FRAUD_PALETTE[0], FRAUD_PALETTE[1]]):
        patch.set_facecolor(colour)
        patch.set_alpha(0.75)
    for median_line in bp["medians"]:
        median_line.set_color("white")
        median_line.set_linewidth(2.5)

    ax_box.axhline(0, color="black", linestyle="--", linewidth=1.2,
                   alpha=0.7, label="Zero  (identity holds)")
    ax_box.set_title(f"{ecol}\nBox Plot  (outliers hidden)", fontweight="bold")
    ax_box.set_ylabel("Error Value")
    ax_box.legend(fontsize=8)

    # Annotate medians directly on the plot
    for i, (vals, colour) in enumerate([(legit_vals, FRAUD_PALETTE[0]),
                                         (fraud_vals, FRAUD_PALETTE[1])], 1):
        med = float(np.median(vals))
        ax_box.text(i + 0.28, med, f"{med:,.0f}", va="center",
                    fontsize=9, color=colour, fontweight="bold")

    # ── Right: log-y histogram ────────────────────────────────────────────────
    ax_hist = axes[row_idx, 1]
    clip_bound = float(np.percentile(np.abs(_eda[ecol].values), 99.5))

    for label, vals, lname in [(0, legit_vals, "Legitimate"),
                                (1, fraud_vals, "Fraud")]:
        ax_hist.hist(
            np.clip(vals, -clip_bound, clip_bound),
            bins=80,
            alpha=0.60,
            color=FRAUD_PALETTE[label],
            density=True,
            label=f"{lname}  (n={len(vals):,})",
        )

    ax_hist.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax_hist.set_yscale("log")
    ax_hist.set_xlabel(f"{ecol}  (clipped to ±99.5th percentile)")
    ax_hist.set_ylabel("Density  (log scale)")
    ax_hist.set_title(f"{ecol}\nDistribution  (log y-scale)", fontweight="bold")
    ax_hist.legend(fontsize=9)

plt.suptitle(
    "Balance Error Features: Fraud vs Legitimate\n"
    "Legit ≈ zero (accounting holds) | Fraud spreads wide (accounting breaks)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
cashout = _eda[_eda["type"] == "CASH_OUT"]

n_legit_plot = 3000
n_fraud_plot = min(len(cashout[cashout["isFraud"] == 1]), 1000)

legit_sample = cashout[cashout["isFraud"] == 0].sample(n=n_legit_plot, random_state=RANDOM_STATE)
fraud_sample = cashout[cashout["isFraud"] == 1].sample(n=n_fraud_plot, random_state=RANDOM_STATE)
plot_df_co   = pd.concat([legit_sample, fraud_sample])

fig, ax = plt.subplots(figsize=(9, 7))

for label, name in [(0, "Legitimate"), (1, "Fraud")]:
    sub = plot_df_co[plot_df_co["isFraud"] == label]
    ax.scatter(
        sub["oldbalanceOrg"],
        sub["newbalanceOrig"],
        c=FRAUD_PALETTE[label],
        label=f"{name}  (n={len(sub):,}  —  sampled for display)",
        alpha=0.30 if label == 0 else 0.80,
        s=7   if label == 0 else 28,
        zorder=1 if label == 0 else 2,
    )

# Reference line y = x (balance completely unchanged — a visual anchor only)
max_bal = float(plot_df_co["oldbalanceOrg"].quantile(0.98))
ax.plot([0, max_bal], [0, max_bal], "k--", linewidth=1, alpha=0.30,
        label="y = x  (no movement — reference)")

ax.set_xlabel("oldbalanceOrg  — origin balance  BEFORE  transaction", fontsize=11)
ax.set_ylabel("newbalanceOrig  — origin balance  AFTER  transaction", fontsize=11)
ax.set_title(
    "CASH_OUT: Origin Balance — Before vs After\n"
    "Legitimate (blue) follows the expected diagonal.\n"
    "Fraud (red) collapses to y ≈ 0 — account fully drained.",
    fontweight="bold", fontsize=11
)
ax.legend(markerscale=2.5, fontsize=9)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
transfer = _eda[_eda["type"] == "TRANSFER"]

n_legit_plot_t = 3000
n_fraud_plot_t = min(len(transfer[transfer["isFraud"] == 1]), 1000)

legit_sample_t = transfer[transfer["isFraud"] == 0].sample(n=n_legit_plot_t, random_state=RANDOM_STATE)
fraud_sample_t = transfer[transfer["isFraud"] == 1].sample(n=n_fraud_plot_t, random_state=RANDOM_STATE)
plot_df_tf     = pd.concat([legit_sample_t, fraud_sample_t])

fig, ax = plt.subplots(figsize=(9, 7))

for label, name in [(0, "Legitimate"), (1, "Fraud")]:
    sub = plot_df_tf[plot_df_tf["isFraud"] == label]
    ax.scatter(
        sub["oldbalanceDest"],
        sub["newbalanceDest"],
        c=FRAUD_PALETTE[label],
        label=f"{name}  (n={len(sub):,}  —  sampled for display)",
        alpha=0.30 if label == 0 else 0.80,
        s=7   if label == 0 else 28,
        zorder=1 if label == 0 else 2,
    )

max_bal_t = float(plot_df_tf["oldbalanceDest"].quantile(0.97))
ax.plot([0, max_bal_t], [0, max_bal_t], "k--", linewidth=1, alpha=0.30,
        label="y = x  (no change — reference)")

ax.set_xlabel("oldbalanceDest  — destination balance  BEFORE  transaction", fontsize=11)
ax.set_ylabel("newbalanceDest  — destination balance  AFTER  transaction", fontsize=11)
ax.set_title(
    "TRANSFER: Destination Balance — Before vs After\n"
    "Legitimate (blue) shows balance growing with transfer.\n"
    "Fraud (red) clusters at (0, 0) — pass-through mule accounts.",
    fontweight="bold", fontsize=11
)
ax.legend(markerscale=2.5, fontsize=9)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
flag_patterns = [
    ("A: newbalanceOrig == 0\n   (origin drained)",   "newbalanceOrig"),
    ("B: oldbalanceOrg == 0\n   (origin pre-empty)",  "oldbalanceOrg"),
    ("C: oldbalanceDest == 0\n   (dest pre-empty)",   "oldbalanceDest"),
    ("D: newbalanceDest == 0\n   (dest post-empty)",  "newbalanceDest"),
]

legit_rates, fraud_rates, flag_labels = [], [], []

for flag_label, flag_col in flag_patterns:
    r_legit = (_eda[_eda["isFraud"] == 0][flag_col] == 0).mean() * 100
    r_fraud  = (_eda[_eda["isFraud"] == 1][flag_col] == 0).mean() * 100
    legit_rates.append(r_legit)
    fraud_rates.append(r_fraud)
    flag_labels.append(flag_label)

# Print table
print("Zero-Balance Pattern Rates: Fraud vs Legitimate\n")
print(f"  {'Pattern':<38} {'Legit (%)':>11} {'Fraud (%)':>10} {'Diff (pp)':>10}")
print("  " + "─" * 73)
for fl, lr, fr in zip(flag_labels, legit_rates, fraud_rates):
    fl_clean = fl.replace("\n   ", "  ")
    print(f"  {fl_clean:<38} {lr:>11.2f} {fr:>10.2f} {fr - lr:>+10.2f}")

print("\n  pp = percentage points.  Positive diff = pattern is more common in fraud.")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(flag_labels))
width = 0.36

bars_l = ax.bar(x - width / 2, legit_rates, width,
                label="Legitimate", color=FRAUD_PALETTE[0], alpha=0.87)
bars_f = ax.bar(x + width / 2, fraud_rates, width,
                label="Fraud",      color=FRAUD_PALETTE[1], alpha=0.87)

# Value labels
for bar in bars_l:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{bar.get_height():.1f}%", ha="center", va="bottom",
            fontsize=8, color=FRAUD_PALETTE[0])
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{bar.get_height():.1f}%", ha="center", va="bottom",
            fontsize=8, color=FRAUD_PALETTE[1], fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(flag_labels, fontsize=8.5, multialignment="center")
ax.set_ylabel("Rate  (%)")
ax.set_title(
    "Zero-Balance Patterns: Fraud vs Legitimate\n"
    "Larger gap → stronger discrimination → encode as binary flag feature in Stage 5",
    fontweight="bold"
)
ax.legend(fontsize=10)
ax.yaxis.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols  = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix   = df_clean[numeric_cols].corr().round(3)

# Mask the upper triangle (correlation matrix is symmetric — show once)
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 9},
)
ax.set_title(
    "Correlation Heatmap — Cleaned Numeric Features  (df_clean)\n"
    "Raw balance columns correlate weakly with isFraud — fraud is non-linear.\n"
    "Engineered error features in Stage 5 will score higher.",
    fontweight="bold"
)
plt.tight_layout()
plt.show()

# Focus on isFraud specifically
print("Correlation of each feature with isFraud  (sorted by |r|):\n")
fraud_corr = corr_matrix["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)
for feat, r in fraud_corr.items():
    bar_len = int(abs(r) * 45)
    print(f"  {feat:<22}  r = {r:+.3f}   {'█' * bar_len}")

print()
print("  → Weak correlations confirm the signal is non-linear.")
print("  → This is why we choose XGBoost over Logistic Regression as the main model.")


In [ ]:
_eda["hour_of_day"] = (_eda["step"] % 24).astype("int32")

hourly = (
    _eda.groupby("hour_of_day")["isFraud"]
    .agg(fraud_count="sum", total_count="count")
    .assign(fraud_rate_pct=lambda x: x["fraud_count"] / x["total_count"] * 100)
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top — volume
axes[0].bar(hourly.index, hourly["total_count"],
            color="#90CAF9", alpha=0.85, label="All transactions")
axes[0].bar(hourly.index, hourly["fraud_count"] * 30,
            color=FRAUD_PALETTE[1], alpha=0.90,
            label="Fraud  (×30 scale for visibility)")
axes[0].set_ylabel("Count")
axes[0].set_title("Transaction Volume and Fraud by Hour of Day  (Filtered Data)",
                  fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))

# Bottom — fraud rate
mean_rate = float(hourly["fraud_rate_pct"].mean())
axes[1].plot(hourly.index, hourly["fraud_rate_pct"],
             color=FRAUD_PALETTE[1], linewidth=2.5, marker="o", markersize=4)
axes[1].fill_between(hourly.index, mean_rate, hourly["fraud_rate_pct"],
                     alpha=0.15, color=FRAUD_PALETTE[1])
axes[1].axhline(mean_rate, color="gray", linestyle="--", linewidth=1.5,
                label=f"Mean: {mean_rate:.3f}%")
axes[1].set_xlabel("Hour of Day  (step % 24 — 0 = midnight)")
axes[1].set_ylabel("Fraud Rate  (%)")
axes[1].set_title("Fraud Rate by Hour of Day", fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

rate_std = float(hourly["fraud_rate_pct"].std())
rate_min = float(hourly["fraud_rate_pct"].min())
rate_max = float(hourly["fraud_rate_pct"].max())
print(f"Fraud rate across hours:  min={rate_min:.3f}%  max={rate_max:.3f}%  std={rate_std:.4f}%")
print()
if rate_std < 0.05:
    print("  → Low variation. Time features may add minimal direct signal.")
    print("  → Still include `step` as a feature — let Stage 11 (SHAP) weight it.")
else:
    print("  → Meaningful variation. `hour_of_day` is a worthwhile feature candidate.")

In [ ]:
# This cell crystallises every EDA observation into a concrete engineering
# decision for Stage 5. Every feature listed is motivated by a specific
# plot or statistic from this stage — not guesswork.
#
# ★ = top-priority features; expected highest SHAP importance in Stage 11.
# Plain = supporting features; useful context or moderate signal.

print("=" * 70)
print("  EDA → FEATURE ENGINEERING PLAN  (Stage 5 implementation targets)")
print("=" * 70)
print()
print("  (★ = top-priority features — expected to dominate SHAP importance)")
print()

feature_plan = [
    ("★ errorBalanceOrig",
     "newbalanceOrig + amount - oldbalanceOrg",
     "Cells 11-14: Core fraud fingerprint — accounting identity violation (origin)"),

    ("★ errorBalanceDest",
     "oldbalanceDest + amount - newbalanceDest",
     "Cells 11-15: Mirror on destination side; especially strong in TRANSFER"),

    ("★ flag_newbalanceOrig_zero",
     "(newbalanceOrig == 0).astype(int)",
     "Cells 5, 14, 16: Origin drained to 0 — much higher rate in fraud"),

    ("★ flag_oldbalanceDest_zero",
     "(oldbalanceDest == 0).astype(int)",
     "Cells 5, 15, 16: Dest pre-empty — pass-through mule account pattern"),

    ("  flag_newbalanceDest_zero",
     "(newbalanceDest == 0).astype(int)",
     "Cell 16: Dest still empty after receiving — suspicious pass-through"),

    ("  flag_oldbalanceOrg_zero",
     "(oldbalanceOrg == 0).astype(int)",
     "Cell 16: Origin already empty before tx — pre-drained account"),

    ("  type_CASH_OUT / type_TRANSFER",
     "pd.get_dummies(df_clean['type'])",
     "Cell 4: One-hot of the 2-value type column — no ordinal relationship"),

    ("  amount",
     "keep raw  (XGBoost needs no scaling)",
     "Cell 10: Fraud skews toward larger amounts — useful baseline signal"),

    ("  step  (or  hour_of_day = step % 24)",
     "keep raw  OR  derive hour_of_day",
     "Cell 18: Mild temporal signal; include and let SHAP confirm weight"),

    ("  oldbalanceOrg, oldbalanceDest",
     "keep raw balance columns",
     "Cells 13-15: Weak direct signal; provide context to error features"),
]

print(f"  {'Feature Name':<33} {'Implementation'}")
print("  " + "─" * 68)
for feat_name, formula, rationale in feature_plan:
    print(f"  {feat_name:<33}  {formula}")
    print(f"  {'':33}   → {rationale}")
    print()

print()
print("  EXCLUDED (dropped in Stage 3 — not features):")
print("  ✗ nameOrig, nameDest  → millions of unique IDs; no generalizable signal")
print("  ✗ isFlaggedFraud      → target leakage (proxy label)")
print()
print("  TARGET : isFraud")
print("  METRIC : PR-AUC  (Average Precision)  —  not accuracy, not ROC-AUC alone")


In [ ]:
del _eda   # Remove the temporary EDA scratchpad — df_clean is our source of truth

print("=" * 62)
print("  STAGE 4 COMPLETE — KEY EDA FINDINGS")
print("=" * 62)
print()
print("  df_clean status  :  unchanged and ready for Stage 5")
print(f"  Shape            :  {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"  Fraud count      :  {df_clean['isFraud'].sum():,}  ({df_clean['isFraud'].mean()*100:.4f}%)")
print(f"  Columns          :  {list(df_clean.columns)}")
print()
print("  ┌──────────────────────────────────────────────────────────┐")
print("  │                                                          │")
print("  │  Finding 1: Balance errors are the #1 fraud signal       │")
print("  │    errorBalanceOrig / errorBalanceDest are non-zero      │")
print("  │    for fraud — the accounting identity breaks.           │")
print("  │                                                          │")
print("  │  Finding 2: Zero-balance patterns are strong flags       │")
print("  │    Fraud drains origins to 0; mule destinations          │")
print("  │    show 0 before AND after receiving funds.              │")
print("  │                                                          │")
print("  │  Finding 3: Raw correlations are weak → non-linear       │")
print("  │    Linear correlation misses the fraud signal.           │")
print("  │    XGBoost will capture it; Logistic Regression won't.   │")
print("  │                                                          │")
print("  │  Finding 4: Amount skews higher for fraud                │")
print("  │    Useful but not sufficient as a standalone signal.     │")
print("  │                                                          │")
print("  │  Finding 5: Temporal signal is mild                      │")
print("  │    Include step / hour_of_day; SHAP decides weight.      │")
print("  │                                                          │")
print("  └──────────────────────────────────────────────────────────┘")
print()
print("  NEXT: Stage 5 — Feature Engineering")
print("  (Implement errorBalance*, zero-flag*, one-hot type, hour_of_day)")

In [ ]:
# =============================================================================
# CELL 21 — Save df_clean for Stage 5 Notebook (Parquet Handoff)
# =============================================================================
# df_clean is the filtered, leakage-free DataFrame produced by Stage 3.
# Saving it here means Stage 5 loads in ~1s instead of re-running all cleaning.

PARQUET_CLEAN = "data/paysim_clean.parquet"
df_clean.to_parquet(PARQUET_CLEAN, index=False)

size_mb = os.path.getsize(PARQUET_CLEAN) / 1024**2
print(f"✅ Saved df_clean → {PARQUET_CLEAN}")
print(f"   Rows    : {len(df_clean):,}")
print(f"   Columns : {list(df_clean.columns)}")
print(f"   File    : {size_mb:.1f} MB")
print(f"\n   Open fraud_detection_stage5_6.ipynb to continue.")